# B2 · Psycopg — talking to PostGIS from Python

**What Psycopg is:** the standard Python driver for PostgreSQL (here version 3, imported as `psycopg`). It opens a connection,
sends your SQL text, and brings the rows back as Python values. It knows nothing about geometry: to Psycopg, a geometry is
just bytes or text. You add Shapely to turn those bytes into shapes.

**Daily picture:** a telephone. `connect` dials the database, a `cursor` is the conversation, `execute` is what you say,
`fetchall` is what you hear back, `commit` is "yes, please save that".

**Where this fits in your plan:** GeoPandas and SQLAlchemy use a driver like this underneath. Knowing it helps you debug,
write safe queries (no SQL injection), and load big data fast (COPY).

⚙️ The database from **B0** must be running.

In [ ]:
import sys, warnings
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))      # lets the notebook find the `geotrain` helper
warnings.filterwarnings("ignore")
from geotrain import write_riverton, DATA_DIR
if not (DATA_DIR / "riverton.gpkg").exists():   # build the Riverton data the first time
    write_riverton()
GPKG = DATA_DIR / "riverton.gpkg"
print("Data folder:", DATA_DIR)

In [ ]:
import psycopg
import shapely
import pandas as pd
from geotrain.db import DSN
print("psycopg", psycopg.__version__)

---
## Level 1 — Basics: connect, execute, fetch

**Goal of this level:** send a query and read the answer as Python values; build your own `sql()` helper.

**Daily-life picture:** Calling a shop, asking a question, writing down the answer.

### Exercise 1.1 — Connection and cursor

**Step 1 · Purpose.** What is the **purpose** of `conn.cursor() / cur.execute() / cur.fetchone()`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`with psycopg.connect(DSN) as conn, conn.cursor() as cur:` then `cur.execute("SELECT count(*) FROM houses")`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with psycopg.connect(DSN) as conn, conn.____() as cur:
    cur.____("SELECT count(*) FROM houses")
    print(cur.____())

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** A cursor sends one SQL statement (`execute`) and reads the result rows (`fetchone`, `fetchall`).

**Real life:** Any script that reads numbers from the city database, e.g. a nightly report.

```python
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.execute("SELECT count(*) FROM houses")
    print(cur.fetchone())
```

</details>

### Exercise 1.2 — Many rows and their column names

**Step 1 · Purpose.** What is the **purpose** of `cur.fetchall() / cur.description`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Column names: `[c.name for c in cur.description]`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.execute("SELECT school, capacity, students FROM schools")
    rows = cur.____()
    cols = [c.name for c in cur.____]
pd.DataFrame(rows, columns=cols)

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** `fetchall` returns all rows as a list of tuples; `description` holds the column names and types.

**Real life:** Turning a query result into a table (DataFrame) for a report.

```python
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.execute("SELECT school, capacity, students FROM schools")
    rows = cur.fetchall()
    cols = [c.name for c in cur.description]
pd.DataFrame(rows, columns=cols)
```

</details>

### Exercise 1.3 — Write your own helper

**Step 1 · Purpose.** What is the **purpose** of wrapping connect/execute/fetch into one function?

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Put 1.2 in a function; pass `params` to `cur.execute(query, params)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
def my_sql(query, params=None):
    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        cur.execute(query, ____)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
my_sql("SELECT name, population FROM neighbourhoods ORDER BY population DESC LIMIT 3")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Reuse: one line per query instead of five, and the connection is always closed properly. (This is the `sql()` helper you used in B1.)

**Real life:** Every analyst builds small helpers like this so notebooks stay short and readable.

```python
def my_sql(query, params=None):
    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        cur.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])
my_sql("SELECT name, population FROM neighbourhoods ORDER BY population DESC LIMIT 3")
```

</details>

---
## Level 2 — Core tools: parameters and transactions

**Goal of this level:** pass values safely and control when changes are saved.

**Daily-life picture:** Filling in a form (safe) instead of letting a stranger write on your cheque (unsafe).

### Exercise 2.1 — Parameters: let the driver insert values

**Step 1 · Purpose.** What is the **purpose** of `cur.execute('... WHERE x > %s', (value,))`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Use `%s` as a placeholder (for any type) and pass a **tuple**: `(500,)` — note the comma.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
my_sql("SELECT school, capacity FROM schools WHERE capacity > ____", (____,))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Sends the SQL and the values separately; the driver inserts them safely and with the right type.

**Real life:** A web map where users type a minimum capacity; their input goes into the query safely.

```python
my_sql("SELECT school, capacity FROM schools WHERE capacity > %s", (800,))
```

</details>

### Exercise 2.2 — Why never build SQL with f-strings

**Step 1 · Purpose.** What is the **purpose** of this experiment? What goes wrong with the f-string version?

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Run both. The f-string version returns **every** school; the parameter version returns none, because no school is literally called that.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
user_input = "x' OR '1'='1"
print(len(my_sql(f"SELECT school FROM schools WHERE school = '{user_input}'")), "rows with f-string")
print(len(my_sql("SELECT school FROM schools WHERE school = ____", (user_input,))), "rows with a parameter")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** It shows that pasting user text into SQL lets that text change the query itself (SQL injection). With `%s` the text stays plain data.

**Real life:** A classic attack on websites: a user types `' OR '1'='1` into a search box and sees (or deletes) data they should not.

```python
user_input = "x' OR '1'='1"
print(len(my_sql(f"SELECT school FROM schools WHERE school = '{user_input}'")), "rows with f-string")
print(len(my_sql("SELECT school FROM schools WHERE school = %s", (user_input,))), "rows with a parameter")
```

Table and column **names** cannot be parameters. For those use `psycopg.sql.Identifier` (see the psycopg docs).

</details>

### Exercise 2.3 — Transactions: commit or roll back

**Step 1 · Purpose.** What is the **purpose** of `conn.commit() / conn.rollback()`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Delete all accidents, count (0), then `rollback()` and count again (260). Nothing is lost!

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with psycopg.connect(DSN) as conn:
    conn.execute("DELETE FROM accidents")
    print("inside the transaction:", conn.execute("SELECT count(*) FROM accidents").fetchone()[0])
    conn.____()
    print("after rollback:", conn.execute("SELECT count(*) FROM accidents").fetchone()[0])

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Changes are private until `commit()` saves them; `rollback()` cancels everything since the last commit.

**Real life:** Updating 1 000 parcels: if one update fails, roll back so the table is not left half-changed.

```python
with psycopg.connect(DSN) as conn:
    conn.execute("DELETE FROM accidents")
    print("inside the transaction:", conn.execute("SELECT count(*) FROM accidents").fetchone()[0])
    conn.rollback()
    print("after rollback:", conn.execute("SELECT count(*) FROM accidents").fetchone()[0])
```

The `with psycopg.connect(...)` block **commits** at the end if no error happened, and rolls back if there was an error.

</details>

### Exercise 2.4 — Named parameters

**Step 1 · Purpose.** What is the **purpose** of `%(name)s with a dict`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`%(x)s`, `%(y)s`, `%(r)s` and pass `{"x": ..., "y": ..., "r": ...}`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
my_sql("""
    SELECT count(*) AS houses FROM houses
    WHERE ST_DWithin(geometry, ST_SetSRID(ST_Point(%(x)s, %(y)s), 32633), %(____)s)
""", {"x": 393_000, "y": 5_821_000, "r": ____})

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Placeholders with names, filled from a dictionary — clearer when a value is used several times.

**Real life:** A query 'houses within {radius} m of point ({x}, {y})' where each value has a clear name.

```python
my_sql("""
    SELECT count(*) AS houses FROM houses
    WHERE ST_DWithin(geometry, ST_SetSRID(ST_Point(%(x)s, %(y)s), 32633), %(r)s)
""", {"x": 393_000, "y": 5_821_000, "r": 500})
```

</details>

---
## Level 3 — Combining: geometries in and out

**Goal of this level:** create a spatial table, insert points, and move Shapely shapes in and out as WKB.

**Daily-life picture:** Sending a parcel: pack it (Shapely → bytes), post it (Psycopg), unpack it on the other side (bytes → Shapely).

### Exercise 3.1 — Create a spatial table and insert one point

**Step 1 · Purpose.** What is the **purpose** of `CREATE TABLE ... geometry(Point, 32633)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`geometry(Point, 32633)` = type + SRID. Insert with parameters for x and y.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
with psycopg.connect(DSN) as conn:
    conn.execute("DROP TABLE IF EXISTS trees")
    conn.execute("CREATE TABLE trees (id serial PRIMARY KEY, species text, geom geometry(____, 32633))")
    conn.execute("INSERT INTO trees (species, geom) VALUES (%s, ST_SetSRID(ST_Point(%s, %s), ____))",
                 ("oak", 392_800, 5_821_400))
my_sql("SELECT id, species, ST_AsText(geom) FROM trees")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Creates a table whose column only accepts points in EPSG:32633; `ST_SetSRID(ST_Point(x, y), 32633)` builds such a point.

**Real life:** A new layer for tree-planting requests collected by a web form.

```python
with psycopg.connect(DSN) as conn:
    conn.execute("DROP TABLE IF EXISTS trees")
    conn.execute("CREATE TABLE trees (id serial PRIMARY KEY, species text, geom geometry(Point, 32633))")
    conn.execute("INSERT INTO trees (species, geom) VALUES (%s, ST_SetSRID(ST_Point(%s, %s), 32633))",
                 ("oak", 392_800, 5_821_400))
my_sql("SELECT id, species, ST_AsText(geom) FROM trees")
```

</details>

### Exercise 3.2 — Insert many rows at once

**Step 1 · Purpose.** What is the **purpose** of `cur.executemany(sql, list_of_tuples)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

Build a list of `(species, x, y)` tuples, then `cur.executemany(...)` with the same INSERT as 3.1.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
import random
random.seed(3)
rows = [(random.choice(["oak", "lime", "birch"]), random.uniform(390_000, 396_000), random.uniform(5_818_000, 5_824_000)) for _ in range(200)]
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.____("INSERT INTO trees (species, geom) VALUES (%s, ST_SetSRID(ST_Point(%s, %s), 32633))", rows)
my_sql("SELECT species, count(*) FROM trees GROUP BY species")

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Runs the same statement for every tuple in a list (one round of talking instead of many separate calls).

**Real life:** Loading 200 tree-planting requests from a spreadsheet.

```python
import random
random.seed(3)
rows = [(random.choice(["oak", "lime", "birch"]), random.uniform(390_000, 396_000), random.uniform(5_818_000, 5_824_000)) for _ in range(200)]
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO trees (species, geom) VALUES (%s, ST_SetSRID(ST_Point(%s, %s), 32633))", rows)
my_sql("SELECT species, count(*) FROM trees GROUP BY species")
```

</details>

### Exercise 3.3 — Send a Shapely shape (WKB)

**Step 1 · Purpose.** What is the **purpose** of `ST_GeomFromWKB(%s, 32633) with shapely.to_wkb(geom)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`shapely.to_wkb(area)` gives bytes. Pass them as a parameter to `ST_GeomFromWKB(%s, 32633)`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
study = shapely.Point(393_000, 5_821_000).buffer(1000)
my_sql("SELECT count(*) FROM trees WHERE ST_Within(geom, ST_GeomFromWKB(%s, 32633))", (shapely.____(study),))

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** Packs a Shapely geometry into WKB bytes (a compact binary format) and lets PostGIS unpack it into a geometry.

**Real life:** You built a study area in Shapely and want PostGIS to count the trees inside it.

```python
study = shapely.Point(393_000, 5_821_000).buffer(1000)
my_sql("SELECT count(*) FROM trees WHERE ST_Within(geom, ST_GeomFromWKB(%s, 32633))", (shapely.to_wkb(study),))
```

</details>

### Exercise 3.4 — Get shapes back into Shapely

**Step 1 · Purpose.** What is the **purpose** of `ST_AsBinary(geom) → shapely.from_wkb(bytes)`? Say in one sentence what it does.

**Step 2 · Real life.** Give **one real-life situation** where a planner or analyst would use it.

<details><summary>💡 <b>Step 3 · Hint</b> (open only after you tried)</summary>

`SELECT name, ST_AsBinary(geometry) FROM neighbourhoods`, then `shapely.from_wkb(row[1])`.

</details>

✍️ **Your answers** (double-click this cell and write):

- Purpose: …
- Real life: …

In [ ]:
df = my_sql("SELECT name, ST_AsBinary(geometry) AS wkb FROM neighbourhoods")
df["geom"] = df["wkb"].apply(lambda b: shapely.____(bytes(b)))
df["area_km2"] = df["geom"].apply(lambda g: g.area / 1e6)
df[["name", "area_km2"]]

<details><summary>✅ <b>Solution</b> (compare after you answered)</summary>

**Purpose:** PostGIS packs geometries as WKB; Shapely unpacks them into shapes you can measure or plot.

**Real life:** Pulling the neighbourhood polygons out of the database for a custom calculation in Python.

```python
df = my_sql("SELECT name, ST_AsBinary(geometry) AS wkb FROM neighbourhoods")
df["geom"] = df["wkb"].apply(lambda b: shapely.from_wkb(bytes(b)))
df["area_km2"] = df["geom"].apply(lambda g: g.area / 1e6)
df[["name", "area_km2"]]
```

GeoPandas `read_postgis` (B4) does exactly this for you, for all rows at once.

</details>

---
## Level 4 — Professional: fast, safe and reusable

**Goal of this level:** load big data with COPY, stream big results, and build safe reusable query functions.

**Daily-life picture:** Using a lorry instead of a bicycle when you move a whole house.

### Exercise 4.1 — Load 20 000 points fast with COPY

**Scenario.** Load 20 000 random sensor readings into a new table `readings`. Compare `executemany` with PostgreSQL's `COPY`.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`with cur.copy("COPY readings (value, geom) FROM STDIN") as cp: cp.write_row((v, 'SRID=32633;POINT(x y)'))`. COPY accepts geometry as EWKT text. Time both with `time.perf_counter()`.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
import time
pts = [(random.random() * 50, random.uniform(390_000, 396_000), random.uniform(5_818_000, 5_824_000)) for _ in range(20_000)]
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS readings")
    cur.execute("CREATE TABLE readings (value float, geom geometry(Point, 32633))")
    t = time.perf_counter()
    cur.executemany("INSERT INTO readings VALUES (%s, ST_SetSRID(ST_Point(%s, %s), 32633))", pts[:2000])
    t_many = (time.perf_counter() - t) / 2000 * 20_000          # scaled to 20 000 rows
    cur.execute("TRUNCATE readings")
    t = time.perf_counter()
    with cur.____("COPY readings (value, geom) FROM STDIN") as cp:
        for v, x, y in pts:
            cp.write_row((v, f"SRID=32633;POINT({x} {y})"))
    t_copy = time.perf_counter() - t
print(f"executemany ≈ {t_many:.2f} s, COPY = {t_copy:.2f} s for 20 000 rows")
my_sql("SELECT count(*) FROM readings")

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data engineering (bulk loading)

COPY streams rows in one go and is usually many times faster. For millions of rows, COPY (or `ogr2ogr`) is the professional choice.

```python
import time
pts = [(random.random() * 50, random.uniform(390_000, 396_000), random.uniform(5_818_000, 5_824_000)) for _ in range(20_000)]
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.execute("DROP TABLE IF EXISTS readings")
    cur.execute("CREATE TABLE readings (value float, geom geometry(Point, 32633))")
    t = time.perf_counter()
    cur.executemany("INSERT INTO readings VALUES (%s, ST_SetSRID(ST_Point(%s, %s), 32633))", pts[:2000])
    t_many = (time.perf_counter() - t) / 2000 * 20_000
    cur.execute("TRUNCATE readings")
    t = time.perf_counter()
    with cur.copy("COPY readings (value, geom) FROM STDIN") as cp:
        for v, x, y in pts:
            cp.write_row((v, f"SRID=32633;POINT({x} {y})"))
    t_copy = time.perf_counter() - t
print(f"executemany ≈ {t_many:.2f} s, COPY = {t_copy:.2f} s for 20 000 rows")
my_sql("SELECT count(*) FROM readings")
```

</details>

### Exercise 4.2 — A safe, reusable question: nearest clinic to any address

**Scenario.** Write `nearest_clinic(x, y)` that returns the name of the nearest clinic and the distance in metres, using parameters and the `<->` index operator. Test it for (391 000, 5 823 500).

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

One SQL query with `%s` placeholders; `ORDER BY geometry <-> ST_SetSRID(ST_Point(%s, %s), 32633) LIMIT 1`. Pass x, y twice (or use named parameters).

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
def nearest_clinic(x, y):
    q = """
        SELECT clinic, round(ST_Distance(geometry, p)::numeric) AS dist_m
        FROM clinics, ST_SetSRID(ST_Point(%(x)s, %(y)s), 32633) AS p
        ORDER BY geometry ____ p LIMIT 1
    """
    return my_sql(q, {"x": x, "y": y}).iloc[0].to_dict()
nearest_clinic(391_000, 5_823_500)

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Proximity (reusable service)

This small function is the core of a real 'find my nearest service' web API: safe parameters + an index-backed KNN search.

```python
def nearest_clinic(x, y):
    q = """
        SELECT clinic, round(ST_Distance(geometry, p)::numeric) AS dist_m
        FROM clinics, ST_SetSRID(ST_Point(%(x)s, %(y)s), 32633) AS p
        ORDER BY geometry <-> p LIMIT 1
    """
    return my_sql(q, {"x": x, "y": y}).iloc[0].to_dict()
nearest_clinic(391_000, 5_823_500)
```

</details>

### Exercise 4.3 — Stream a big result without filling memory

**Scenario.** Read all 20 000 readings in batches of 5 000 with a **server-side (named) cursor** and compute the mean value in Python.

**Step 1 · Question type.** What *kind* of question is this? (descriptive, measurement, proximity, overlay, statistical, modelling, decision, temporal…)

**Step 2 · Plan.** Write the commands you need **in order**, in plain words first.

**Step 3 · Code.** Turn the plan into code below.

<details><summary>💡 <b>Hint</b></summary>

`conn.cursor(name="stream")` keeps the result on the server; `cur.fetchmany(5000)` brings one batch at a time. Keep a running sum and count.

</details>

✍️ **Your answers:**

- Question type: …
- Plan (steps): 1. … 2. … 3. …
- Result / interpretation: …

In [ ]:
total, n = 0.0, 0
with psycopg.connect(DSN) as conn, conn.cursor(name="____") as cur:
    cur.execute("SELECT value FROM readings")
    while batch := cur.fetchmany(____):
        total += sum(r[0] for r in batch)
        n += len(batch)
print(n, round(total / n, 2))

<details><summary>✅ <b>Solution</b></summary>

🧭 **Question type:** Data engineering (server-side cursor)

Same idea as chunked file reading in A4: process in pieces. (Of course, `SELECT avg(value)` in SQL is simpler — do the maths in the database when you can.)

```python
total, n = 0.0, 0
with psycopg.connect(DSN) as conn, conn.cursor(name="stream") as cur:
    cur.execute("SELECT value FROM readings")
    while batch := cur.fetchmany(5000):
        total += sum(r[0] for r in batch)
        n += len(batch)
print(n, round(total / n, 2))
```

</details>

---
## 🏁 Final test

Use Psycopg (and Shapely where needed). For each: **plan in words → code → interpretation.**

### Test 1 — 🛠️ Task question

**A.** Write a function `houses_near(x, y, r)` that returns the number of residents within `r` metres of a point. Make it injection-safe.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
def houses_near(x, y, r):
    return my_sql("""SELECT coalesce(sum(residents), 0) AS residents FROM houses
                   WHERE ST_DWithin(geometry, ST_SetSRID(ST_Point(%s, %s), 32633), %s)""", (x, y, r)).iloc[0, 0]
print(houses_near(393_000, 5_821_000, 500))
```

</details>

### Test 2 — 🛠️ Task question

**B.** Insert a new clinic "Harbour Care" at (395 200, 5 823 300) with 5 doctors, check it, then **undo** it with a rollback.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
with psycopg.connect(DSN) as conn:
    conn.execute("INSERT INTO clinics (clinic, doctors, geometry) VALUES (%s, %s, ST_SetSRID(ST_Point(%s, %s), 32633))",
                 ("Harbour Care", 5, 395_200, 5_823_300))
    print(conn.execute("SELECT count(*) FROM clinics").fetchone())
    conn.rollback()
print(my_sql("SELECT count(*) FROM clinics"))
```

</details>

### Test 3 — 🛠️ Task question

**C.** Build a Shapely line from (390 500, 5 819 000) to (395 500, 5 823 000) (a planned tram line), send it to PostGIS, and count accidents within 100 m of it.

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

```python
tram = shapely.LineString([(390_500, 5_819_000), (395_500, 5_823_000)])
print(my_sql("SELECT count(*) FROM accidents WHERE ST_DWithin(geometry, ST_GeomFromWKB(%s, 32633), 100)", (shapely.to_wkb(tram),)))
```

</details>

### Test 4 — 🗺️ Decision question

**D.** Your colleague builds queries like `f"... WHERE name = '{name}'"` in a web app. What do you tell them, in two sentences?

✍️ **Your answer** (plan in words first, then code if the question asks for it):

…

In [ ]:
# your code (if needed)

<details><summary>✅ <b>Model answer</b> (open only after you answered)</summary>

"This allows SQL injection: a user can type text that changes your query and reads or deletes data. Always pass values as parameters (`%s`) and let psycopg insert them."

</details>

---
## 🏗️ Project — A live air-quality feed with Psycopg

**The brief.** Riverton's 18 sensors now send **hourly PM2.5 readings**. Build the pipeline with Psycopg: a readings table with keys, a safe
        insert of a day of data, a fast bulk load of a week, a 'latest reading' layer for QGIS, and an alert query. Then watch QGIS update.

**Your data** (made for this project, in `data/projects/`):

| Layer / file | What it holds |
|---|---|
| `public.sensors` | the 18 sensor locations with a typical PM2.5 level (loaded in B0) |
| `projects.pm_readings` | the table you create: one row per sensor per hour |

**How to work:** for each task write the plan in words, then the code, then one sentence with the result. Model answers are hidden; open them only after you tried. The last part happens in **QGIS**.

In [ ]:
from geotrain.projects import build_projects, PROJ_DIR, publish
build_projects()

import pandas as pd

### Project task 1

**Create the table** `projects.pm_readings (sensor_id, ts, pm25)` with a primary key on (sensor_id, ts) and a CHECK (pm25 ≥ 0).

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
with psycopg.connect(DSN) as conn:
    conn.execute("CREATE SCHEMA IF NOT EXISTS projects")
    conn.execute("DROP TABLE IF EXISTS projects.pm_readings CASCADE")
    conn.execute("""CREATE TABLE projects.pm_readings (sensor_id text NOT NULL, ts timestamp NOT NULL,
                      pm25 real NOT NULL CHECK (pm25 >= 0), PRIMARY KEY (sensor_id, ts))""")
```

</details>

### Project task 2

**One day, safely.** Simulate 24 hourly readings per sensor (typical level + rush-hour bump + noise) and insert with `executemany` and parameters.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
import numpy as np
rng = np.random.default_rng(7)
base = my_sql("SELECT sensor_id, pm25 FROM public.sensors")
def readings(day, hours=24):
    rows = []
    for sid, lvl in zip(base.sensor_id, base.pm25):
        for h in range(hours):
            rush = 6 if h in (7, 8, 17, 18) else 0
            rows.append((sid, day + pd.Timedelta(hours=h), float(max(0, lvl + rush + rng.normal(0, 2)))))
    return rows
day1 = readings(pd.Timestamp("2025-09-01"))
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    cur.executemany("INSERT INTO projects.pm_readings VALUES (%s, %s, %s)", day1)
my_sql("SELECT count(*) FROM projects.pm_readings")
```

</details>

### Project task 3

**A week, fast.** Load the next 7 days with `COPY` and compare the row count. What happens if you run the day-1 insert again (primary key!)?

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
week = [r for d in range(2, 9) for r in readings(pd.Timestamp(f"2025-09-{d:02d}"))]
with psycopg.connect(DSN) as conn, conn.cursor() as cur:
    with cur.copy("COPY projects.pm_readings (sensor_id, ts, pm25) FROM STDIN") as cp:
        for r in week:
            cp.write_row(r)
print(my_sql("SELECT count(*) FROM projects.pm_readings"))
try:
    with psycopg.connect(DSN) as conn, conn.cursor() as cur:
        cur.executemany("INSERT INTO projects.pm_readings VALUES (%s, %s, %s)", day1[:1])
except psycopg.errors.UniqueViolation as e:
    print("refused duplicate:", e.diag.message_primary)
```

</details>

### Project task 4

**Latest reading as a map layer.** A view `projects.pm_latest` with each sensor's most recent reading and its geometry (`DISTINCT ON`), and a view of daily means.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
with psycopg.connect(DSN) as conn:
    conn.execute("""CREATE OR REPLACE VIEW projects.pm_latest AS
        SELECT DISTINCT ON (r.sensor_id) r.sensor_id, r.ts, r.pm25, s.geometry
        FROM projects.pm_readings r JOIN public.sensors s USING (sensor_id)
        ORDER BY r.sensor_id, r.ts DESC""")
    conn.execute("""CREATE OR REPLACE VIEW projects.pm_daily AS
        SELECT sensor_id, ts::date AS day, round(avg(pm25)::numeric, 1) AS mean_pm25 FROM projects.pm_readings GROUP BY 1, 2""")
my_sql("SELECT sensor_id, ts, round(pm25::numeric, 1) AS pm25 FROM projects.pm_latest ORDER BY pm25 DESC LIMIT 3")
```
`DISTINCT ON (sensor_id) ... ORDER BY sensor_id, ts DESC` keeps the first (= newest) row per sensor — a PostgreSQL speciality.

</details>

### Project task 5

**Alert function.** `alerts(limit)` returns the sensors whose **daily mean** exceeded the limit on any day, with the number of days (safe parameter). Try the WHO guideline, 15 µg/m³.

✍️ **Your plan and result:** …

In [ ]:
# your code

<details><summary>✅ <b>Model answer</b></summary>

```python
def alerts(limit):
    return my_sql("""SELECT sensor_id, count(*) AS days_over, max(mean_pm25) AS worst
                      FROM projects.pm_daily WHERE mean_pm25 > %s GROUP BY sensor_id ORDER BY days_over DESC""", (limit,))
alerts(15)
```

</details>

### 🗺️ Project in QGIS

1. Load **pm_latest** (id `sensor_id`) and style it *Graduated* on `pm25` (green → red). Label with `round("pm25", 1)`.
2. Run task 3 again with the next week (change the dates) in the notebook, then in QGIS press **F5 / refresh**: the map shows the new latest values. That is a *live* layer.
3. Load **pm_daily** as a table and use the *Data Plotly* plugin (optional) to draw one sensor's daily means.
4. Layer Properties → *Rendering → Refresh layer at interval* (e.g. 10 s) makes QGIS poll the view automatically.

<details><summary>🔌 <b>How to connect QGIS to PostGIS</b> (once per computer)</summary>

1. Start the database (Docker: `docker start geotrain-db`).
2. In QGIS, open the **Browser** panel → right-click **PostgreSQL** → **New Connection…**
3. Name `geotrain`, Host `localhost`, Port `5432`, Database `geotrain`.
   Authentication → *Basic*: user `geo`, password `geo` (tick *Store*). Click **Test Connection** → OK.
4. Expand **geotrain → projects** (the schema) and **drag a table onto the map**.
5. To run SQL and see the result on the map: **Database → DB Manager → PostGIS → geotrain → SQL Window**.
   Write the query → **Execute** → tick **Load as new layer**, choose the geometry column and a unique id column → **Load**.
6. To open a GeoPackage instead: drag the `.gpkg` file from your file manager into QGIS.

</details>

### 📦 Deliverables (tick when done)

- [ ] Table `pm_readings` with keys and 8 days of data
- [ ] Views `pm_latest` and `pm_daily`
- [ ] Alert table for 15 µg/m³ + QGIS live map

### 🪞 Reflection — your long-term plan

Will you mostly *read* from a database someone else runs, or *build* your own? Which Psycopg skills matter most for that (parameters, COPY, transactions)?

✍️ **Your answer:** …